# Agentspace integration

Last Update:

- Sept 12, 2025
- ADK version: agnostic

Prerequisite:

- You have already deployed ADK on Agent Engine
- You have enabled Discovery Engine API
- You have granted Vertex AI user role on the discovery engine service account
- Your GCP is allowlisted for Agentspace custom agent integration (ask your FSR/ CE)

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.sandbox.google.com/github/hupili/google-adk-101/blob/main/Agentspace_integration.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Run in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fhupili%2Fgoogle-adk-101%2Fmain%2FAgentspace_integration.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-enterprise-logo.png" alt="Google Cloud Colab Enterprise logo"><br> Run in Colab Enterprise
    </a>
  </td>    
  <td style="text-align: center">
    <a href="https://github.com/hupili/google-adk-101/blob/main/Agentspace_integration.ipynb">
      <img width="32px" src="https://www.gstatic.com/monospace/240802/git_host_github_mask.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/hupili/google-adk-101%2Fmain%2FAgentspace_integration.ipynb">
      <img width="32px" src="https://www.gstatic.com/cloud/images/navigation/vertex-ai.svg" alt="Vertex AI logo"><br> Open in Vertex AI Workbench
    </a>
  </td>
</table>

## Environment Setup

In [1]:
import os

# API_KEY = '' # @param {type:"string"}
# os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE" # Use Vertex AI API
# os.environ["GOOGLE_API_KEY"] = API_KEY

PROJECT_ID = "hupili-genai-bb"  # @param {type:"string"}
if not PROJECT_ID:
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

LOCATION = "us-central1"  # @param {type:"string"}

STAGING_BUCKET = "gs://agent_engine_deploy_staging"  # @param {type:"string"}

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"  # Use Vertex AI API

# [your-project-id]

In [2]:
import vertexai

vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=STAGING_BUCKET,
)

In [3]:
from google.colab import auth

auth.authenticate_user()

In [4]:
!pip install google-adk==1.8.0

In [5]:
from google import adk

adk.__version__

'1.8.0'

In [6]:
import subprocess
import json
import requests

# --- Configuration ---
PROJECT_ID = "hupili-genai-bb"
PROJECT_NUMBER = "160214671203"
AS_APP = "googlebbastest_1745397917180"
# This is the displayName for the assistant itself, not the individual agents
ASSISTANT_LEVEL_DISPLAY_NAME = "My Awesome Custom Agents"


# --- Helper function to get gcloud access token ---
def get_gcloud_access_token():
    """Fetches the gcloud access token."""
    try:
        # Ensure gcloud is available and user is authenticated
        process = subprocess.run(
            ["gcloud", "auth", "print-access-token"],
            capture_output=True,
            text=True,
            check=True,
        )
        return process.stdout.strip()
    except subprocess.CalledProcessError as e:
        print(f"Error getting gcloud access token: {e}")
        print(f"Stderr: {e.stderr}")
        return None
    except FileNotFoundError:
        print(
            "Error: 'gcloud' command not found. Make sure the Google Cloud SDK is installed and in your PATH."
        )
        return None


# --- Main Logic ---
def update_assistant_with_agent_configs(agent_configs_data):
    access_token = get_gcloud_access_token()
    if not access_token:
        print("Failed to retrieve access token. Exiting.")
        return

    # Construct the API URL
    # The URL points to 'default_assistant' as per your curl command
    api_url = (
        f"https://discoveryengine.googleapis.com/v1alpha/projects/{PROJECT_NUMBER}/"
        f"locations/global/collections/default_collection/engines/{AS_APP}/"
        f"assistants/default_assistant"
    )

    # Set query parameters (for the updateMask)
    query_params = {
        "updateMask": "agent_configs"  # As per your curl command's ?updateMask=agent_configs
    }

    # Construct headers
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
        "x-goog-user-project": PROJECT_ID,  # Important for Google Cloud APIs
    }

    # Build the "agentConfigs" part of the payload dynamically from our list
    formatted_agent_configs_payload = []
    for agent_data in agent_configs_data:
        formatted_agent_configs_payload.append(
            {
                "id": agent_data["id"],
                "displayName": agent_data["display_name"],
                "vertexAiSdkAgentConnectionInfo": {
                    "reasoningEngine": agent_data["reasoning_engine"]
                },
                "toolDescription": agent_data["description"],
                "icon": {"uri": agent_data["icon_uri"]},
            }
        )

    # Construct the full request payload
    # The 'name' field in the payload should match the resource being patched.
    payload = {
        "name": (
            f"projects/{PROJECT_NUMBER}/locations/global/collections/"
            f"default_collection/engines/{AS_APP}/assistants/default_assistant"
        ),
        "displayName": ASSISTANT_LEVEL_DISPLAY_NAME,  # displayName for the assistant
        "agentConfigs": formatted_agent_configs_payload,
    }

    print(f"Sending PATCH request to: {api_url}")
    print(f"With query params: {query_params}")
    # print(f"With headers: {headers}") # Token will be printed, maybe omit for security
    print(f"With payload: {json.dumps(payload, indent=2)}")

    # Make the PATCH request
    try:
        response = requests.patch(
            api_url,
            headers=headers,
            params=query_params,  # requests library handles adding this to the URL
            data=json.dumps(payload),  # Body data must be a JSON string
        )

        # Raise an exception for bad status codes (4xx or 5xx)
        response.raise_for_status()

        print(f"\nSuccess! Status Code: {response.status_code}")
        print("Response JSON:")
        # Attempt to print JSON, but handle cases where response might be empty or not JSON
        try:
            print(json.dumps(response.json(), indent=2))
        except json.JSONDecodeError:
            print(f"Response content (not valid JSON): {response.text}")

    except requests.exceptions.HTTPError as e:
        print(f"\nHTTP Error: {e}")
        print(f"Response status code: {e.response.status_code}")
        print(f"Response content: {e.response.text}")
    except requests.exceptions.RequestException as e:
        # For other network issues, timeouts, etc.
        print(f"\nRequest failed: {e}")

In [11]:
# Abstracted agent configurations as a list of dictionaries
# Each dictionary represents one agent's configuration
agent_configs_data = [
    {
        "id": "dice_agent",
        "display_name": "Yo! Good Dice Agent",
        "description": "The agent rolls dice and tracks the roll results.",
        "reasoning_engine": f"projects/{PROJECT_NUMBER}/locations/us-central1/reasoningEngines/6087608854902734848",
        "icon_uri": "https://fonts.gstatic.com/s/i/short-term/release/googlesymbols/corporate_fare/default/24px.svg",
    },
    {
        "id": "llm_auditor",
        "display_name": "Yo! Nice LLM Auditor",
        "description": "fact check anything.",
        "reasoning_engine": f"projects/{PROJECT_NUMBER}/locations/us-central1/reasoningEngines/7916070303615156224",
        "icon_uri": "https://fonts.gstatic.com/s/i/short-term/release/googlesymbols/corporate_fare/default/24px.svg",
    },
    {
        "id": "simple_search",
        "display_name": "Yo! Simple search",
        "description": "just simply search anything.",
        "reasoning_engine": f"projects/{PROJECT_NUMBER}/locations/us-central1/reasoningEngines/8759404518212370432",
        "icon_uri": "https://fonts.gstatic.com/s/i/short-term/release/googlesymbols/corporate_fare/default/24px.svg",
    },
]

update_assistant_with_agent_configs(agent_configs_data)

Sending PATCH request to: https://discoveryengine.googleapis.com/v1alpha/projects/160214671203/locations/global/collections/default_collection/engines/googlebbastest_1745397917180/assistants/default_assistant
With query params: {'updateMask': 'agent_configs'}
With payload: {
  "name": "projects/160214671203/locations/global/collections/default_collection/engines/googlebbastest_1745397917180/assistants/default_assistant",
  "displayName": "My Awesome Custom Agents",
  "agentConfigs": [
    {
      "id": "dice_agent",
      "displayName": "Yo! Good Dice Agent",
      "vertexAiSdkAgentConnectionInfo": {
        "reasoningEngine": "projects/160214671203/locations/us-central1/reasoningEngines/6087608854902734848"
      },
      "toolDescription": "The agent rolls dice and tracks the roll results.",
      "icon": {
        "uri": "https://fonts.gstatic.com/s/i/short-term/release/googlesymbols/corporate_fare/default/24px.svg"
      }
    },
    {
      "id": "llm_auditor",
      "displayName"